In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import glob
from pathlib import Path
from tqdm import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
import math
import random
import matplotlib.pyplot as plt

In [ ]:


# --- 1. CONFIGURATION ---
# Define hyperparameters and file paths in a single, clean dictionary.
CONFIG = {
    # Data Paths
    'base_data_path': '/content/drive/MyDrive/Google_AI_Studio/ARCAGI2025/data',
    'input_directory': 'GridTransitionDataset/training_transformed_unique_ids',
    'output_dir': 'training_results', # New output directory for models and plots

    # Model Hyperparameters
    'vocab_size': 12,
    'max_seq_len': 1802,
    'd_model': 512,
    'nhead': 8,
    'num_layers': 6,
    'dim_feedforward': 2048,
    'dropout': 0.1,

    # Training Parameters
    'batch_size': 32,
    'learning_rate_g': 0.0002,
    'learning_rate_d': 0.0002,
    'num_epochs': 5,
    'n_splits': 5,
    'train_generator_every': 2,
}


In [ ]:

# --- 2. UTILITY CLASS FOR SAVING ---
class DriveSaver:
    """
    A utility class to handle saving models and files to Google Drive.
    """
    def __init__(self, base_path):
        self.base_path = Path(base_path)

    def save_model(self, model, file_name, sub_dir='models'):
        """
        Saves a PyTorch model's state dictionary to the specified path.
        """
        save_dir = self.base_path / sub_dir
        save_dir.mkdir(parents=True, exist_ok=True)
        save_path = save_dir / file_name
        torch.save(model.state_dict(), save_path)
        print(f"Model saved to {save_path}")

    def save_plot(self, fig, file_name, sub_dir='plots'):
        """
        Saves a Matplotlib figure to the specified path.
        """
        save_dir = self.base_path / sub_dir
        save_dir.mkdir(parents=True, exist_ok=True)
        save_path = save_dir / file_name
        fig.savefig(save_path)
        print(f"Plot saved to {save_path}")


In [ ]:

# --- 3. DATASET CLASS ---
class ARCDataset(Dataset):
    """
    Custom Dataset class for loading and preprocessing ARC data.
    """
    def __init__(self, file_paths, max_seq_len):
        self.file_paths = file_paths
        self.max_seq_len = max_seq_len
        self.processed_sequences = self._load_data()

    def _load_data(self, verbose=False):
        """
        Loads sequences from JSON files based on the specified format.
        Gracefully handles and skips files with unexpected errors.
        """
        sequences = []
        skipped_count = 0
        for file_path in tqdm(self.file_paths, desc="Loading data files"):
            try:
                with open(file_path, 'r') as f:
                    data = json.load(f)
                    sequence = data['input'][0]
                    sequences.append(sequence)
            except Exception as e:
                skipped_count += 1
                if verbose:
                    tqdm.write(f"Skipping file: {file_path} due to an unexpected error: {e}")

        if verbose:
            tqdm.write(f"\nTotal files skipped: {skipped_count}")
        return sequences

    def __len__(self):
        return len(self.processed_sequences)

    def __getitem__(self, idx):
        # Pad or truncate the sequence to the fixed max_seq_len
        sequence = self.processed_sequences[idx]
        if len(sequence) > self.max_seq_len:
            sequence = sequence[:self.max_seq_len]
        else:
            sequence.extend([0] * (self.max_seq_len - len(sequence)))

        return torch.tensor(sequence, dtype=torch.long)


In [ ]:

# --- 4. MODEL ARCHITECTURE ---
class PositionalEncoding(nn.Module):
    """
    Standard Positional Encoding for Transformer models.
    """
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return x

class Generator(nn.Module):
    """
    The Generator (Policy Network) in the RL framework.
    """
    def __init__(self, config):
        super(Generator, self).__init__()
        self.config = config
        self.embedding = nn.Embedding(config['vocab_size'], config['d_model'])
        self.pos_encoder = PositionalEncoding(config['d_model'], config['max_seq_len'])
        self.transformer_encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=config['d_model'],
                nhead=config['nhead'],
                dim_feedforward=config['dim_feedforward'],
                dropout=config['dropout'],
                batch_first=True
            ),
            num_layers=config['num_layers']
        )
        self.fc_out = nn.Linear(config['d_model'], config['vocab_size'])

    def forward(self, x):
        # The forward pass now returns logits for sampling
        x = self.embedding(x)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        logits = self.fc_out(x)
        return logits

class Discriminator(nn.Module):
    """
    The Discriminator (Reward Function) in the RL framework.
    """
    def __init__(self, config):
        super(Discriminator, self).__init__()
        self.config = config

        self.embedding_layer = nn.Embedding(config['vocab_size'], config['d_model'])
        self.pos_encoder = PositionalEncoding(config['d_model'], config['max_seq_len'])

        self.transformer_encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=config['d_model'],
                nhead=config['nhead'],
                dim_feedforward=config['dim_feedforward'],
                dropout=config['dropout'],
                batch_first=True
            ),
            num_layers=config['num_layers']
        )
        self.fc_out = nn.Linear(config['d_model'], 1)

    def forward(self, x):
        embedded_x = self.embedding_layer(x)
        embedded_x = self.pos_encoder(embedded_x)
        x = self.transformer_encoder(embedded_x)
        x = torch.mean(x, dim=1)
        output = self.fc_out(x)
        return output


In [ ]:

# --- 5. RL TRAINER CLASS ---
class RLTrainer:
    def __init__(self, config):
        self.config = config
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Using device: {self.device}")
        self.saver = DriveSaver(Path(config['base_data_path']) / config['output_dir'])

    def train_one_fold(self, fold_idx, train_loader):
        """
        Trains the Generator (as a policy network) and Discriminator for one fold.
        """
        print(f"\n--- Starting Fold {fold_idx + 1}/{self.config['n_splits']} ---")

        generator = Generator(self.config).to(self.device)
        discriminator = Discriminator(self.config).to(self.device)

        criterion_d = nn.BCEWithLogitsLoss()
        optimizer_G = optim.Adam(generator.parameters(), lr=self.config['learning_rate_g'])
        optimizer_D = optim.Adam(discriminator.parameters(), lr=self.config['learning_rate_d'])

        fold_disc_losses_epoch = []
        fold_gen_rewards_epoch = []

        for epoch in range(self.config['num_epochs']):
            disc_losses_batch = []
            gen_rewards_batch = []

            for i, real_sequences in enumerate(tqdm(train_loader, desc=f"Fold {fold_idx + 1}, Epoch {epoch + 1}")):
                real_sequences = real_sequences.to(self.device)
                batch_size = real_sequences.size(0)

                # --- Train Discriminator ---
                discriminator.zero_grad()

                # Train with real sequences
                real_labels = torch.ones(batch_size, 1, device=self.device)
                d_output_real = discriminator(real_sequences)
                d_loss_real = criterion_d(d_output_real, real_labels)
                d_loss_real.backward()

                # Generate fake sequences for D training
                noise_input = torch.randint(0, self.config['vocab_size'], (batch_size, self.config['max_seq_len']), device=self.device)

                with torch.no_grad():
                    fake_sequences_logits = generator(noise_input)
                    # Sample hard tokens from the Generator for the Discriminator
                    fake_sequences = torch.argmax(F.log_softmax(fake_sequences_logits, dim=-1), dim=-1)

                # Train with fake sequences
                fake_labels = torch.zeros(batch_size, 1, device=self.device)
                d_output_fake = discriminator(fake_sequences)
                d_loss_fake = criterion_d(d_output_fake, fake_labels)
                d_loss_fake.backward()

                d_loss = d_loss_real + d_loss_fake
                optimizer_D.step()

                disc_losses_batch.append(d_loss.item())

                # --- Train Generator with REINFORCE (Policy Gradient) ---
                if i % self.config['train_generator_every'] == 0:
                    generator.zero_grad()

                    # Generate sequences
                    noise_input = torch.randint(0, self.config['vocab_size'], (batch_size, self.config['max_seq_len']), device=self.device)
                    g_output_logits = generator(noise_input)

                    # Compute log probabilities of the sampled tokens
                    log_probs = F.log_softmax(g_output_logits, dim=-1)

                    # Sample tokens from the distribution
                    sampled_tokens = torch.multinomial(torch.exp(log_probs.view(-1, self.config['vocab_size'])), 1).view(batch_size, self.config['max_seq_len'])

                    # Use the Discriminator to get rewards for the sampled sequences
                    # The `detach()` is crucial to prevent gradients from flowing into the D during G's update
                    rewards = discriminator(sampled_tokens).detach()

                    # Calculate REINFORCE loss: -log(P) * R
                    # We use the log_probs of the actual sampled tokens
                    gathered_log_probs = log_probs.gather(dim=-1, index=sampled_tokens.unsqueeze(-1)).squeeze(-1)
                    g_loss = -torch.mean(gathered_log_probs * rewards)

                    g_loss.backward()
                    optimizer_G.step()
                    gen_rewards_batch.append(rewards.mean().item())

            avg_disc_loss_epoch = sum(disc_losses_batch) / len(disc_losses_batch)
            avg_gen_reward_epoch = sum(gen_rewards_batch) / len(gen_rewards_batch)
            fold_disc_losses_epoch.append(avg_disc_loss_epoch)
            fold_gen_rewards_epoch.append(avg_gen_reward_epoch)

            print(f"Epoch {epoch+1} finished. Avg Discriminator Loss: {avg_disc_loss_epoch:.4f}, Avg Generator Reward: {avg_gen_reward_epoch:.4f}")

        # Use the new DriveSaver class to save models
        self.saver.save_model(generator, f'generator_fold_{fold_idx+1}.pth')
        self.saver.save_model(discriminator, f'discriminator_fold_{fold_idx+1}.pth')

        avg_fold_discriminator_loss = sum(fold_disc_losses_epoch) / len(fold_disc_losses_epoch)
        avg_fold_generator_reward = sum(fold_gen_rewards_epoch) / len(fold_gen_rewards_epoch)

        print(f"--- Finished Fold {fold_idx + 1} ---")
        print(f"Average Discriminator Loss for Fold {fold_idx + 1}: {avg_fold_discriminator_loss:.4f}")
        print(f"Average Generator Reward for Fold {fold_idx + 1}: {avg_fold_generator_reward:.4f}")

        return fold_disc_losses_epoch, fold_gen_rewards_epoch

    def plot_results(self, all_d_losses, all_g_rewards):
        """
        Generates and saves plots of the training metrics using the DriveSaver.
        """
        # Plotting Discriminator Loss
        plt.figure(figsize=(10, 6))
        for i, fold_losses in enumerate(all_d_losses):
            plt.plot(fold_losses, label=f'Fold {i+1}')
        plt.title('Discriminator Loss over Epochs (Per Fold)')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True)
        self.saver.save_plot(plt, 'discriminator_loss.png')
        plt.close()

        # Plotting Generator Reward
        plt.figure(figsize=(10, 6))
        for i, fold_rewards in enumerate(all_g_rewards):
            plt.plot(fold_rewards, label=f'Fold {i+1}')
        plt.title('Generator Reward over Epochs (Per Fold)')
        plt.xlabel('Epoch')
        plt.ylabel('Reward')
        plt.legend()
        plt.grid(True)
        self.saver.save_plot(plt, 'generator_reward.png')
        plt.close()


In [1]:

# --- 6. MAIN EXECUTION ---
if __name__ == '__main__':
    data_dir = os.path.join(CONFIG['base_data_path'], CONFIG['input_directory'])
    all_files = glob.glob(os.path.join(data_dir, '*.json'))

    if not all_files:
        print(f"Error: No JSON files found in directory: {data_dir}")
        print("Please check your file path and ensure the directory is not empty.")
    else:
        print(f"Found {len(all_files)} files. Proceeding with K-Fold cross-validation.")

        kf = KFold(n_splits=CONFIG['n_splits'], shuffle=True, random_state=42)

        all_fold_discriminator_losses = []
        all_fold_generator_rewards = []

        trainer = RLTrainer(CONFIG)

        for fold, (train_index, val_index) in enumerate(kf.split(all_files)):
            train_file_paths = [all_files[i] for i in train_index]

            train_dataset = ARCDataset(train_file_paths, CONFIG['max_seq_len'])
            train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)

            d_losses_per_epoch, g_rewards_per_epoch = trainer.train_one_fold(fold, train_loader)

            all_fold_discriminator_losses.append(d_losses_per_epoch)
            all_fold_generator_rewards.append(g_rewards_per_epoch)

        print("\n--- Cross-Validation Results ---")
        average_discriminator_loss_cv = np.mean([np.mean(losses) for losses in all_fold_discriminator_losses])
        average_generator_reward_cv = np.mean([np.mean(rewards) for rewards in all_fold_generator_rewards])

        print(f"Average Discriminator Loss across {CONFIG['n_splits']} folds: {average_discriminator_loss_cv:.4f}")
        print(f"Average Generator Reward across {CONFIG['n_splits']} folds: {average_generator_reward_cv:.4f}")

        # Generate and save plots after training is complete
        trainer.plot_results(all_fold_discriminator_losses, all_fold_generator_rewards)


Mounted at /content/drive
Found 16144 files. Proceeding with K-Fold cross-validation.
Using device: cuda


Loading data files:   1%|          | 118/12915 [00:50<1:31:33,  2.33it/s]


KeyboardInterrupt: 